In [ ]:
import re
import pandas as pd
import numpy as np
from collections import Counter
# 1. Load Dataset
file = "training.1600000.processed.noemoticon.csv"

df = pd.read_csv(
    file,
    encoding="latin-1",
    header=None
)

df.columns = [
    "target",
    "id",
    "date",
    "flag",
    "user",
    "text"
]

print("Dataset loaded successfully!")
print(df.head())

# Use first 100,000 tweets for faster processing
df = df.head(100000)

# 2. Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_text"] = df["text"].apply(clean_text)

print("\nCleaned Data:")
print(df[["text", "cleaned_text"]].head())

# 3. Tokenization
sentences = []

for text in df["cleaned_text"]:
    words = text.split()

    if len(words) > 0:
        sentences.append(words)

# 4. Create Bigram Counts
bigram_counts = Counter()

for words in sentences:
    for i in range(len(words) - 1):
        bigram = (words[i], words[i + 1])
        bigram_counts[bigram] += 1

# 5. Create Trigram Counts
trigram_counts = Counter()

for words in sentences:
    for i in range(len(words) - 2):
        trigram = (
            words[i],
            words[i + 1],
            words[i + 2]
        )
        trigram_counts[trigram] += 1

print("\nTotal Bigrams:", len(bigram_counts))
print("Total Trigrams:", len(trigram_counts))

# 6. Next Word Prediction
def predict_next_word(text):

    text = clean_text(text)
    words = text.split()

    if len(words) == 0:
        return "Please enter some words"

    # Trigram prediction
    if len(words) >= 2:

        last_two = (words[-2], words[-1])

        candidates = {}

        for trigram, count in trigram_counts.items():

            if (
                trigram[0] == last_two[0]
                and
                trigram[1] == last_two[1]
            ):
                candidates[trigram[2]] = count

        if candidates:
            return max(
                candidates,
                key=candidates.get
            )

    # Bigram prediction
    last_word = words[-1]

    candidates = {}

    for bigram, count in bigram_counts.items():

        if bigram[0] == last_word:
            candidates[bigram[1]] = count

    if candidates:
        return max(
            candidates,
            key=candidates.get
        )

    return "No prediction available"

# 7. Multiple Suggestions
def suggest_words(text, n=3):

    text = clean_text(text)
    words = text.split()

    if len(words) == 0:
        return []

    candidates = Counter()

    # Trigram suggestions
    if len(words) >= 2:

        last_two = (words[-2], words[-1])

        for trigram, count in trigram_counts.items():

            if (
                trigram[0] == last_two[0]
                and
                trigram[1] == last_two[1]
            ):
                candidates[trigram[2]] += count

    # Bigram suggestions
    if not candidates:

        last_word = words[-1]

        for bigram, count in bigram_counts.items():

            if bigram[0] == last_word:
                candidates[bigram[1]] += count

    return [
        word
        for word, count in candidates.most_common(n)
    ]

# 8. Test Sentences
test_sentences = [
    "i want",
    "thank you",
    "how are",
    "good morning",
    "i love"
]

print("\n========================================")
print("NEXT-WORD PREDICTIONS")
print("========================================")

for sentence in test_sentences:

    prediction = predict_next_word(sentence)

    print("\nInput:", sentence)
    print("Predicted Word:", prediction)

# 9. Autocomplete Suggestions
print("\n========================================")
print("AUTOCOMPLETE SUGGESTIONS")
print("========================================")

for sentence in test_sentences:

    suggestions = suggest_words(sentence, 3)

    print("\nInput:", sentence)
    print("Suggestions:", suggestions)

# 10. User Input
print("\n========================================")
print("MOBILE KEYBOARD / CHAT AUTOCOMPLETE")
print("========================================")

while True:

    query = input("\nEnter text (or type exit): ")

    if query.lower() == "exit":
        break

    prediction = predict_next_word(query)
    suggestions = suggest_words(query, 3)

    print("Predicted Next Word:", prediction)
    print("Autocomplete Suggestions:", suggestions)

# 11. Save Cleaned Dataset
df.to_csv(
    "Sentiment140_Cleaned.csv",
    index=False
)

print("\nCleaned dataset saved successfully!")

Dataset loaded successfully!
   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  

Cleaned Data:
                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he c